In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


In [ ]:
import os
import pandas as pd
import scipy.io
import anndata
import scanpy as sc

In [ ]:
adata = sc.read_h5ad(str(PROJECT_ROOT / "zhangzemin/GSE243013_NSCLC_immune_scRNA_data.h5ad"))

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scipy.sparse  # 新增导入



# ====================
# 第一步：数据预处理
# ====================

# 1.1 创建样本到治疗反应的映射字典
response_mapping = (
    adata.obs[['sampleID', 'pathological_response']]
    .drop_duplicates()
    .set_index('sampleID')['pathological_response']
    .to_dict()
)

# 1.2 添加响应分组列到观测数据中
adata.obs['response_group'] = (
    adata.obs['sampleID']
    .map(response_mapping)
    .replace({
        'pCR': 'responder',
        'MPR': 'responder',
        'non-MPR': 'non-responder'
    })
)

# 1.3 过滤掉未知样本
adata = adata[adata.obs['pathological_response'] != 'unknow'].copy()

# ====================
# 第二步：基因表达分析
# ====================

# 2.1 提取目标基因数据
target_genes = ['EPAS1',  'TEAD1']

# 检查基因是否存在
missing_genes = [gene for gene in target_genes if gene not in adata.var_names]
if missing_genes:
    raise ValueError(f"以下基因不存在于数据集中: {missing_genes}")

# 2.2 创建包含基因表达和分组信息的数据框
expr_df = pd.DataFrame(
    data=adata[:, target_genes].X.toarray() if scipy.sparse.issparse(adata.X) else adata[:, target_genes].X,
    columns=target_genes,
    index=adata.obs_names
).join(adata.obs[['response_group']])

# ====================
# 第三步：统计分析
# ====================

# 3.1 执行Mann-Whitney U检验
results = {}
for gene in target_genes:
    group1 = expr_df.loc[expr_df['response_group'] == 'responder', gene]
    group2 = expr_df.loc[expr_df['response_group'] == 'non-responder', gene]
    
    # 执行统计检验
    stat, pval = stats.mannwhitneyu(group1, group2, alternative='two-sided')
    
    # 计算效应量
    n1, n2 = len(group1), len(group2)
    eff_size = stat / (n1 * n2)  # 计算Cliff's delta的近似值
    
    results[gene] = {
        'statistic': stat,
        'p_value': pval,
        'effect_size': eff_size,
        'responder_mean': group1.mean(),
        'non_responder_mean': group2.mean()
    }

# 打印结果
print("="*50)
print("差异表达分析结果:")
for gene, res in results.items():
    print(f"\n基因 {gene}:")
    print(f"p值: {res['p_value']:.3e}")
    print(f"效应量: {res['effect_size']:.3f}")
    print(f"响应组均值: {res['responder_mean']:.3f}")
    print(f"非响应组均值: {res['non_responder_mean']:.3f}")
print("="*50)

# ====================
# 第四步：可视化
# ====================

# 4.1 设置绘图样式
sns.set(style="whitegrid", font_scale=1.2)
plt.figure(figsize=(12, 6))

# 4.2 绘制分面箱线图
palette = {'responder': '#2c7bb6', 'non-responder': '#d7191c'}

for i, gene in enumerate(target_genes, 1):
    plt.subplot(1, len(target_genes), i)
    sns.boxplot(
        x='response_group',
        y=gene,
        data=expr_df,
        order=['responder', 'non-responder'],
        palette=palette,
        showfliers=True,
        width=0.6
    )
    
    # 添加统计注释
    max_y = expr_df[gene].max() * 1.1
    plt.plot([0, 1], [max_y]*2, color='black', lw=1.5)
    plt.text(0.5, max_y*1.05, 
             f"p = {results[gene]['p_value']:.2e}\n" +
             f"Δ = {results[gene]['effect_size']:.2f}",
             ha='center', va='bottom')
    
    plt.title(gene, weight='bold')
    plt.xlabel('')
    plt.ylabel('Expression Level')
    plt.ylim(bottom=0)

plt.tight_layout()
plt.show()

# # ====================
# # 第五步：高级分析（可选）
# # ====================
# # 如果需要更严格的差异分析，可以使用scanpy的rank_genes_groups
# sc.tl.rank_genes_groups(
#     adata,
#     groupby='response_group',
#     groups=['responder'],
#     reference='non-responder',
#     method='wilcoxon'
# )

# # 提取目标基因的结果
# print("\nScanpy差异分析结果:")
# result_df = sc.get.rank_genes_groups_df(adata, group='responder')
# display(result_df[result_df['names'].isin(target_genes)])

In [ ]:
for gene in ['EPAS1', 'TEAD1']:
    print(f"\n===== {gene} =====")
    print(
        expr_df.groupby('response_group')[gene]
        .apply(lambda x: pd.Series({
            'n_cells': len(x),
            'n_nonzero': int((x > 0).sum()),
            'pct_nonzero': float((x > 0).mean() * 100),
            'mean': float(x.mean()),
            'median': float(x.median()),
            'max': float(x.max())
        }))
    )

是在做 sample-level pseudobulk（样本层面伪批量）比较：
先把同一样本内所有细胞的 counts 加总，再按文库大小标准化成 logCPM，最后比较 responder 和 non-responder 两组样本之间的差异。

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

SAMPLE_COL = "sampleID"
RAW_RESPONSE_COL = "pathological_response"
GROUP_COL = "response_group"
TARGET_GENES = ["EPAS1", "OAT","FASN","PPA1","ACSL3","ACSL4","PGS1","HSD17B4"]
PREFERRED_LAYER = "counts"

response_map = {
    "pCR": "responder",
    "MPR": "responder",
    "non-MPR": "non-responder"
}

if PREFERRED_LAYER in adata.layers:
    X = adata.layers[PREFERRED_LAYER]
    var_names = pd.Index(adata.var_names.astype(str))
    matrix_source = f"adata.layers['{PREFERRED_LAYER}']"
elif adata.raw is not None:
    X = adata.raw.X
    var_names = pd.Index(adata.raw.var_names.astype(str))
    matrix_source = "adata.raw.X"
else:
    raise ValueError(
        "没有找到可用于 true pseudobulk 的原始 counts。\n"
        "你当前 adata.X 看起来已经做过 log1p，不能直接拿来做 true pseudobulk。\n"
        "请先检查 adata.layers['counts'] 或 adata.raw 是否保存了原始 counts。"
    )

print("Using matrix:", matrix_source)

missing_genes = [g for g in TARGET_GENES if g not in var_names]
if missing_genes:
    raise ValueError(f"以下基因不在表达矩阵里: {missing_genes}")

obs_full = adata.obs[[SAMPLE_COL, RAW_RESPONSE_COL]].copy()
obs_full[GROUP_COL] = obs_full[RAW_RESPONSE_COL].map(response_map)

valid_mask = (
    obs_full[SAMPLE_COL].notna()
    & (obs_full[SAMPLE_COL].astype(str) != "")
    & obs_full[GROUP_COL].notna()
)

obs_use = obs_full.loc[valid_mask, [SAMPLE_COL, GROUP_COL]].copy()
obs_idx = np.where(valid_mask.to_numpy())[0]

sample_group_check = obs_use.groupby(SAMPLE_COL)[GROUP_COL].nunique()
bad_samples = sample_group_check[sample_group_check > 1].index.tolist()
if len(bad_samples) > 0:
    raise ValueError(
        "以下 sampleID 对应了多个 response_group，请先清理样本注释不一致问题:\n"
        + "\n".join(map(str, bad_samples[:20]))
    )

sample_to_group = (
    obs_use[[SAMPLE_COL, GROUP_COL]]
    .drop_duplicates()
    .set_index(SAMPLE_COL)[GROUP_COL]
)

X_use = X[obs_idx]
sample_series = obs_use[SAMPLE_COL].astype(str)

if sp.issparse(X_use):
    cell_library = np.asarray(X_use.sum(axis=1)).ravel()
else:
    cell_library = np.asarray(X_use.sum(axis=1)).ravel()

library_size = pd.Series(cell_library, index=sample_series).groupby(level=0).sum()
n_cells = sample_series.value_counts().reindex(library_size.index)

pb = pd.DataFrame(index=library_size.index)
pb["n_cells"] = n_cells.astype(int)
pb["library_size"] = library_size.astype(float)
pb[GROUP_COL] = sample_to_group.reindex(pb.index)

for gene in TARGET_GENES:
    gene_idx = var_names.get_loc(gene)

    if sp.issparse(X_use):
        gene_cell = np.asarray(X_use[:, gene_idx].toarray()).ravel()
    else:
        gene_cell = np.asarray(X_use[:, gene_idx]).ravel()

    gene_count = pd.Series(gene_cell, index=sample_series).groupby(level=0).sum()
    pb[f"{gene}_count"] = gene_count.reindex(pb.index).fillna(0).astype(float)
    pb[f"{gene}_logCPM"] = np.log1p(
        pb[f"{gene}_count"] / pb["library_size"].replace(0, np.nan) * 1e6
    )

print(pb[[GROUP_COL, "n_cells"] + [f"{g}_count" for g in TARGET_GENES] + [f"{g}_logCPM" for g in TARGET_GENES]].head())

stats_res = {}
for gene in TARGET_GENES:
    ycol = f"{gene}_logCPM"
    g1 = pb.loc[pb[GROUP_COL] == "responder", ycol].dropna()
    g2 = pb.loc[pb[GROUP_COL] == "non-responder", ycol].dropna()

    if len(g1) == 0 or len(g2) == 0:
        stats_res[gene] = {
            "p_value": np.nan,
            "responder_mean": np.nan,
            "non_responder_mean": np.nan,
            "n_responder": len(g1),
            "n_non_responder": len(g2)
        }
    else:
        _, pval = mannwhitneyu(g1, g2, alternative="two-sided")
        stats_res[gene] = {
            "p_value": pval,
            "responder_mean": g1.mean(),
            "non_responder_mean": g2.mean(),
            "n_responder": len(g1),
            "n_non_responder": len(g2)
        }

print("\nSample-level pseudobulk results")
for gene, res in stats_res.items():
    print(f"\n{gene}")
    print(f"  n_responder      = {res['n_responder']}")
    print(f"  n_non_responder  = {res['n_non_responder']}")
    print(f"  responder_mean   = {res['responder_mean']:.4f}")
    print(f"  non_responder_mean = {res['non_responder_mean']:.4f}")
    print(f"  p_value          = {res['p_value']:.3e}")

sns.set(style="whitegrid", font_scale=1.7)
palette = {"responder": "#2c7bb6", "non-responder": "#d7191c"}

fig, axes = plt.subplots(1, len(TARGET_GENES), figsize=(6 * len(TARGET_GENES), 5.8))
if len(TARGET_GENES) == 1:
    axes = [axes]

for ax, gene in zip(axes, TARGET_GENES):
    ycol = f"{gene}_logCPM"
    sub = pb[[GROUP_COL, ycol]].dropna().copy()

    sns.boxplot(
        x=GROUP_COL,
        y=ycol,
        data=sub,
        order=["responder", "non-responder"],
        palette=palette,
        showfliers=False,
        width=0.5,
        ax=ax
    )

    sns.stripplot(
        x=GROUP_COL,
        y=ycol,
        data=sub,
        order=["responder", "non-responder"],
        palette=palette,
        size=6,
        jitter=0.15,
        edgecolor="black",
        linewidth=0.4,
        ax=ax
    )

    ymin = sub[ycol].min()
    ymax = sub[ycol].max()
    yrange = ymax - ymin
    if yrange == 0:
        yrange = 1

    # 给显著性横线和p值预留更合理的空间
    yline = ymax + 0.08 * yrange
    ytext = ymax + 0.13 * yrange

    ax.plot([0, 1], [yline, yline], color="black", lw=1.2)
    ax.text(
        0.5,
        ytext,
        f"p = {stats_res[gene]['p_value']:.2e}",
        ha="center",
        va="bottom",
        fontsize=17
    )

    # 给顶部留白，避免和标题重叠
    ax.set_ylim(ymin - 0.05 * yrange, ymax + 0.22 * yrange)

    # 标题往上挪一点
    ax.set_title(gene, fontweight="bold", pad=12)

    ax.set_xlabel("")
    ax.set_ylabel("Sample pseudobulk logCPM")

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
import os

plt.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig("sample_pseudobulk_boxplot.pdf", format="pdf", bbox_inches="tight")
print("Saved to:", os.path.abspath("sample_pseudobulk_boxplot.pdf"))
plt.show()

In [ ]:
中止

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
from scipy import sparse

gene_list = [
    'ITM2A', 'KLRB1', 'SPOCK2', 'ETS1', 'IL32', 'RORA', 'LCK', 'CD2', 'TNFRSF4', 'CD3E',
    'CD69', 'CD247', 'TNFRSF18', 'CD96', 'CD7', 'SKAP1', 'LTB', 'GIMAP7', 'SYNE2', 'GATA3',
    'TNFRSF25', 'PIK3IP1', 'IL7R', 'LBH', 'EVL', 'ITK', 'CD3D', 'ZAP70', 'ACAP1', 'LEF1',
    'TUBA4A', 'PBXIP1', 'PYHIN1', 'PRKCH', 'FYN', 'SH2D1A', 'RHOH', 'CD27', 'LAT', 'GZMM',
    'CLEC2D', 'TIGIT', 'ARID5B', 'ISG20', 'RASGRP1', 'TRAT1', 'CD40LG', 'CD3G', 'BTG1', 'CD6'
]

if "sampleID" not in adata.obs.columns:
    raise KeyError("adata.obs 中没有 'sampleID' 这一列。")

missing_genes = [g for g in gene_list if g not in adata.var_names]
if missing_genes:
    print("警告：以下基因不存在，将排除：")
    print(missing_genes)

gene_list = [g for g in gene_list if g in adata.var_names]
if len(gene_list) == 0:
    raise ValueError("gene_list 里的基因一个都不在 adata.var_names 中，无法打分。")

X = adata.X
if sparse.issparse(X):
    X = X.tocsr()
else:
    X = sparse.csr_matrix(X)

sample_series = adata.obs["sampleID"].astype(str)
sample_names = pd.Index(sample_series.unique())
sample_codes = pd.Categorical(sample_series, categories=sample_names).codes

n_cells = adata.n_obs
n_samples = len(sample_names)

cell_sample_mat = sparse.csr_matrix(
    (
        np.ones(n_cells, dtype=np.float32),
        (np.arange(n_cells), sample_codes)
    ),
    shape=(n_cells, n_samples)
)

X_pseudobulk = cell_sample_mat.T @ X

obs0 = adata.obs.copy()
obs0["sampleID"] = obs0["sampleID"].astype(str)

sample_obs_list = []
for sid in sample_names:
    sub = obs0.loc[obs0["sampleID"] == sid]
    row = {"sampleID": sid, "n_cells": sub.shape[0]}
    for col in obs0.columns:
        if col == "sampleID":
            continue
        vals = sub[col].dropna()
        row[col] = vals.iloc[0] if len(vals) > 0 else np.nan
    sample_obs_list.append(row)

sample_obs = pd.DataFrame(sample_obs_list).set_index("sampleID")

adata = ad.AnnData(
    X=X_pseudobulk,
    obs=sample_obs,
    var=adata.var.copy()
)

sc.tl.score_genes(
    adata,
    gene_list=gene_list,
    ctrl_size=len(gene_list),
    score_name="my_gene_score",
    use_raw=False,
    random_state=0
)

print(adata)
print(adata.obs[["n_cells", "my_gene_score"]].head())

In [ ]:
adata.obs

In [ ]:
# 设置 Pandas 的显示选项，确保所有内容都能打印出来
pd.set_option('display.max_rows', None)  # 设置最大显示行数为 None，即显示所有行
pd.set_option('display.max_colwidth', None)  # 设置最大列宽为 None，即显示所有列内容

print(adata.obs['pre_treatment_staging'].value_counts())

# 恢复默认的显示设置
pd.reset_option('display.max_rows')
pd.reset_option('display.max_colwidth')

In [ ]:
# 提取obs信息并保存为CSV文件
obs_df = adata.obs
obs_df.to_csv(str(PROJECT_ROOT / f"泛癌S/张泽民PD1治疗肺癌数据验证modul1/GSE243013_NSCLC_immune_scRNA_data_normalize_打分_module.csv"), index=True)

In [ ]:
adata.obs['my_gene_score']

In [ ]:
print(adata.obs['response_group'].value_counts())

In [ ]:
print(adata.obs['pathological_response'].value_counts())

In [ ]:
print(adata.obs['pathological_response_rate'].value_counts())

In [ ]:
# 设置 Pandas 的显示选项，确保所有内容都能打印出来
pd.set_option('display.max_rows', None)  # 设置最大显示行数为 None，即显示所有行
pd.set_option('display.max_colwidth', None)  # 设置最大列宽为 None，即显示所有列内容

print(adata.obs['cancer_type'].value_counts())

# 恢复默认的显示设置
pd.reset_option('display.max_rows')
pd.reset_option('display.max_colwidth')

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ====================================
# 新增功能：按分期分组分析（含均值标注）
# ====================================

# 1. 获取所有肿瘤分期（排除未知分期）
valid_stages = [
    stage for stage, count in adata.obs['cancer_type'].value_counts().items() 
    if stage != "unknowm"
]
print(f"将分析以下分期: {valid_stages}")

# 2. 定义统计分析函数（新增均值标注）
def perform_analysis(sub_adata, stage_name):
    """在指定分期的数据子集上执行分析"""
    # 提取评分和分组数据
    score_df = pd.DataFrame({
        'gene_score': sub_adata.obs['my_gene_score'],
        'response_group': sub_adata.obs['response_group']
    })
    
    # 检查组别是否存在
    groups_present = score_df['response_group'].unique()
    if len(groups_present) < 2:
        print(f"跳过 {stage_name}：缺少responder/non-responder分组")
        return None
    
    # 统计检验
    try:
        responder = score_df[score_df['response_group'] == 'responder']['gene_score']
        non_responder = score_df[score_df['response_group'] == 'non-responder']['gene_score']
        
        # 确保每组至少有5个样本
        if len(responder) <5 or len(non_responder)<5:
            print(f"跳过 {stage_name}：样本量不足（R:{len(responder)}, NR:{len(non_responder)}）")
            return None
            
        stat, pval = stats.mannwhitneyu(responder, non_responder, alternative='two-sided')
        eff_size = stat / (len(responder) * len(non_responder))  # Cliff's delta
    except Exception as e:
        print(f"{stage_name} 分析失败: {str(e)}")
        return None
    
    # 可视化设置
    plt.figure(figsize=(6, 4))
    sns.set(style="whitegrid", font_scale=1.0)
    
    # 创建小提琴图
    ax = sns.violinplot(
        x='response_group',
        y='gene_score',
        data=score_df,
        order=['responder', 'non-responder'],
        palette={'responder': '#DC0000FF',
        'non-responder': '#3C5488FF'},
        inner="quartile",
        cut=0
    )
    
    # ====================
    # 新增：均值标注部分
    # ====================
    for i, group in enumerate(['responder', 'non-responder']):
        mean_val = score_df[score_df['response_group'] == group]['gene_score'].mean()
        # 绘制均值点
        ax.scatter(
            i, mean_val, 
            color='white',          # 填充色
            s=100,                  # 点大小
            zorder=3,               # 显示层级
            edgecolor='black',      # 边缘色
            linewidth=1.5           # 边缘线宽
        )
        # 添加均值文本
        ax.text(
            i,                     # x位置
            mean_val,              # y位置
            f'{mean_val:.2f}',     # 保留两位小数
            ha='center',           # 水平居中
            va='bottom',           # 垂直对齐于底部
            color='black',         # 文字颜色
            fontweight='bold'      # 粗体显示
        )
    
    # 统计显著性标注
    max_y = score_df['gene_score'].max() * 1.15
    ax.plot([0, 1], [max_y]*2, color='black', lw=1)
    
    def get_stars(p):
        """将P值转换为星号表示"""
        if p < 0.001: return '***'
        elif p < 0.01: return '**'
        elif p < 0.05: return '*'
        else: return 'NS'
    
    ax.text(
        0.5,                          # x居中
        max_y*1.05,                   # y位置
        f"{get_stars(pval)}",         # 显著性符号
        ha='center',                  # 水平居中
        va='bottom',                  # 垂直对齐
        fontsize=12                   # 字体大小
    )
    
    # 图表美化
    plt.title(
        f"{stage_name}",  # 标题包含样本量  f"{stage_name} (n={len(sub_adata)})",
        weight='bold', 
        pad=15
    )
    plt.xlabel('')
    plt.ylabel('Gene Set Activity Score', labelpad=10)
    plt.ylim(score_df['gene_score'].min()*0.9, max_y*1.1)
    
    # 保存图片
    plt.savefig(
        str(PROJECT_ROOT / f"泛癌S/张泽民PD1治疗肺癌数据验证modul1/GeneScore_Comparison_{stage_name}.pdf"), 
        bbox_inches='tight',  # 自动裁剪白边
        dpi=300               # 高分辨率输出
    )
    plt.show()
    plt.close()
    
    return {
        'stage': stage_name,
        'responder_mean': responder.mean(),
        'non_responder_mean': non_responder.mean(),
        'p_value': pval,
        'effect_size': eff_size,
        'n_responder': len(responder),
        'n_non_responder': len(non_responder)
    }

# 3. 循环执行每个分期的分析
results = []
for stage in valid_stages:
    # 创建分期的数据子集
    stage_mask = adata.obs['cancer_type'] == stage
    sub_adata = adata[stage_mask].copy()
    
    # 执行分析
    result = perform_analysis(sub_adata, stage)
    if result:
        results.append(result)

# 4. 汇总结果并保存
result_df = pd.DataFrame(results)
print("\n统计分析汇总:")
print(result_df.to_string(index=False))  # 优化表格显示

# 保存详细结果
result_df.to_csv(
    str(PROJECT_ROOT / "泛癌S/张泽民PD1治疗肺癌数据验证modul1/Stratified_Analysis_Results.csv"), 
    index=False,          # 不保存行索引
    float_format='%.4f'   # 保留4位小数
)


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
import re

outdir = str(PROJECT_ROOT / "泛癌S/张泽民PD1治疗肺癌数据验证modul1")
os.makedirs(outdir, exist_ok=True)

exclude_stages = {"unknowm", "unknown", "UNKNOWM", "UNKNOWN", "nan", "None", ""}

valid_df = adata.obs[["pre_treatment_staging", "cancer_type", "response_group", "my_gene_score"]].copy()
valid_df["pre_treatment_staging"] = valid_df["pre_treatment_staging"].astype(str)
valid_df["cancer_type"] = valid_df["cancer_type"].astype(str)
valid_df["response_group"] = valid_df["response_group"].astype(str)

valid_df = valid_df[
    (~valid_df["pre_treatment_staging"].isin(exclude_stages)) &
    (~valid_df["cancer_type"].isin(["nan", "None", ""])) &
    (~valid_df["response_group"].isin(["nan", "None", ""]))
].copy()

cancer_types = ["LUAD", "LUSC"]
valid_df = valid_df[valid_df["cancer_type"].isin(cancer_types)].copy()

print("将分析以下癌种和分期组合：")
for cancer in cancer_types:
    stages_this_cancer = valid_df.loc[
        valid_df["cancer_type"] == cancer, "pre_treatment_staging"
    ].value_counts().index.tolist()
    print(f"{cancer}: {stages_this_cancer}")

def safe_name(x):
    return re.sub(r"[\\/*?:\"<>| ]+", "_", str(x))

def perform_analysis(sub_adata, cancer_name, stage_name):
    score_df = pd.DataFrame({
        "gene_score": sub_adata.obs["my_gene_score"].astype(float),
        "response_group": sub_adata.obs["response_group"].astype(str)
    })

    score_df = score_df[
        score_df["response_group"].isin(["responder", "non-responder"])
    ].copy()

    groups_present = score_df["response_group"].unique()
    if len(groups_present) < 2:
        print(f"跳过 {cancer_name} - {stage_name}：缺少 responder/non-responder 分组")
        return None

    responder = score_df.loc[score_df["response_group"] == "responder", "gene_score"]
    non_responder = score_df.loc[score_df["response_group"] == "non-responder", "gene_score"]

    if len(responder) < 5 or len(non_responder) < 5:
        print(f"跳过 {cancer_name} - {stage_name}：样本量不足（R:{len(responder)}, NR:{len(non_responder)}）")
        return None

    try:
        stat, pval = stats.mannwhitneyu(
            responder,
            non_responder,
            alternative="two-sided"
        )
        eff_size = stat / (len(responder) * len(non_responder))
    except Exception as e:
        print(f"{cancer_name} - {stage_name} 分析失败: {str(e)}")
        return None

    sns.set(style="whitegrid", font_scale=1.0)
    plt.figure(figsize=(6, 4))

    ax = sns.violinplot(
        x="response_group",
        y="gene_score",
        data=score_df,
        order=["responder", "non-responder"],
        palette={
            "responder": "#DC0000FF",
            "non-responder": "#3C5488FF"
        },
        inner="quartile",
        cut=0
    )

    for i, group in enumerate(["responder", "non-responder"]):
        mean_val = score_df.loc[score_df["response_group"] == group, "gene_score"].mean()
        ax.scatter(
            i,
            mean_val,
            color="white",
            s=100,
            zorder=3,
            edgecolor="black",
            linewidth=1.5
        )
        ax.text(
            i,
            mean_val,
            f"{mean_val:.2f}",
            ha="center",
            va="bottom",
            color="black",
            fontweight="bold"
        )

    def get_stars(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return "NS"

    y_min = score_df["gene_score"].min()
    y_max = score_df["gene_score"].max()
    y_range = y_max - y_min if y_max > y_min else 1.0

    line_y = y_max + y_range * 0.10
    text_y = y_max + y_range * 0.16
    upper_lim = y_max + y_range * 0.25
    lower_lim = y_min - y_range * 0.08

    ax.plot([0, 1], [line_y, line_y], color="black", lw=1)
    ax.text(
        0.5,
        text_y,
        get_stars(pval),
        ha="center",
        va="bottom",
        fontsize=12
    )

    plt.title(
        f"{cancer_name} - {stage_name}",
        weight="bold",
        pad=15
    )
    plt.xlabel("")
    plt.ylabel("Gene Set Activity Score", labelpad=10)
    plt.ylim(lower_lim, upper_lim)

    pdf_name = f"GeneScore_Comparison_{safe_name(cancer_name)}_{safe_name(stage_name)}.pdf"
    plt.savefig(
        os.path.join(outdir, pdf_name),
        bbox_inches="tight",
        dpi=300
    )
    plt.show()
    plt.close()

    return {
        "cancer_type": cancer_name,
        "stage": stage_name,
        "responder_mean": responder.mean(),
        "non_responder_mean": non_responder.mean(),
        "p_value": pval,
        "effect_size": eff_size,
        "n_responder": len(responder),
        "n_non_responder": len(non_responder)
    }

results = []

for cancer in cancer_types:
    cancer_mask = adata.obs["cancer_type"].astype(str) == cancer
    cancer_adata = adata[cancer_mask].copy()

    stage_list = [
        stage for stage, count in cancer_adata.obs["pre_treatment_staging"].astype(str).value_counts().items()
        if stage not in exclude_stages
    ]

    print(f"\n开始分析 {cancer}，分期包括：{stage_list}")

    for stage in stage_list:
        stage_mask = cancer_adata.obs["pre_treatment_staging"].astype(str) == stage
        sub_adata = cancer_adata[stage_mask].copy()

        result = perform_analysis(sub_adata, cancer, stage)
        if result is not None:
            results.append(result)

result_df = pd.DataFrame(results)

print("\n统计分析汇总：")
if result_df.shape[0] > 0:
    print(result_df.to_string(index=False))
else:
    print("没有可输出的结果，可能所有分组都因样本量不足或缺少组别被跳过。")

result_df.to_csv(
    os.path.join(outdir, "Stratified_Analysis_Results_ByCancerType_AndStage.csv"),
    index=False,
    float_format="%.4f"
)

In [ ]:
adata.obs

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ====================================
# 新增功能：按分期分组分析（含均值标注）
# ====================================

# 1. 获取所有肿瘤分期（排除未知分期）
valid_stages = [
    stage for stage, count in adata.obs['gender'].value_counts().items() 
    if stage != "unknowm"
]
print(f"将分析以下分期: {valid_stages}")

# 2. 定义统计分析函数（新增均值标注）
def perform_analysis(sub_adata, stage_name):
    """在指定分期的数据子集上执行分析"""
    # 提取评分和分组数据
    score_df = pd.DataFrame({
        'gene_score': sub_adata.obs['my_gene_score'],
        'response_group': sub_adata.obs['response_group']
    })
    
    # 检查组别是否存在
    groups_present = score_df['response_group'].unique()
    if len(groups_present) < 2:
        print(f"跳过 {stage_name}：缺少responder/non-responder分组")
        return None
    
    # 统计检验
    try:
        responder = score_df[score_df['response_group'] == 'responder']['gene_score']
        non_responder = score_df[score_df['response_group'] == 'non-responder']['gene_score']
        
        # 确保每组至少有5个样本
        if len(responder) <5 or len(non_responder)<5:
            print(f"跳过 {stage_name}：样本量不足（R:{len(responder)}, NR:{len(non_responder)}）")
            return None
            
        stat, pval = stats.mannwhitneyu(responder, non_responder, alternative='two-sided')
        eff_size = stat / (len(responder) * len(non_responder))  # Cliff's delta
    except Exception as e:
        print(f"{stage_name} 分析失败: {str(e)}")
        return None
    
    # 可视化设置
    plt.figure(figsize=(6, 4))
    sns.set(style="whitegrid", font_scale=1.0)
    
    # 创建小提琴图
    ax = sns.violinplot(
        x='response_group',
        y='gene_score',
        data=score_df,
        order=['responder', 'non-responder'],
        palette={'responder': '#DC0000FF',
        'non-responder': '#3C5488FF'},
        inner="quartile",
        cut=0
    )
    
    # ====================
    # 新增：均值标注部分
    # ====================
    for i, group in enumerate(['responder', 'non-responder']):
        mean_val = score_df[score_df['response_group'] == group]['gene_score'].mean()
        # 绘制均值点
        ax.scatter(
            i, mean_val, 
            color='white',          # 填充色
            s=100,                  # 点大小
            zorder=3,               # 显示层级
            edgecolor='black',      # 边缘色
            linewidth=1.5           # 边缘线宽
        )
        # 添加均值文本
        ax.text(
            i,                     # x位置
            mean_val,              # y位置
            f'{mean_val:.2f}',     # 保留两位小数
            ha='center',           # 水平居中
            va='bottom',           # 垂直对齐于底部
            color='black',         # 文字颜色
            fontweight='bold'      # 粗体显示
        )
    
    # 统计显著性标注
    max_y = score_df['gene_score'].max() * 1.15
    ax.plot([0, 1], [max_y]*2, color='black', lw=1)
    
    def get_stars(p):
        """将P值转换为星号表示"""
        if p < 0.001: return '***'
        elif p < 0.01: return '**'
        elif p < 0.05: return '*'
        else: return 'NS'
    
    ax.text(
        0.5,                          # x居中
        max_y*1.05,                   # y位置
        f"{get_stars(pval)}",         # 显著性符号
        ha='center',                  # 水平居中
        va='bottom',                  # 垂直对齐
        fontsize=12                   # 字体大小
    )
    
    # 图表美化
    plt.title(
        f"{stage_name}",  # 标题包含样本量  f"{stage_name} (n={len(sub_adata)})",
        weight='bold', 
        pad=15
    )
    plt.xlabel('')
    plt.ylabel('Gene Set Activity Score', labelpad=10)
    plt.ylim(score_df['gene_score'].min()*0.9, max_y*1.1)
    
    # 保存图片
    plt.savefig(
        f"GeneScore_Comparison_{stage_name}.pdf", 
        bbox_inches='tight',  # 自动裁剪白边
        dpi=300               # 高分辨率输出
    )
    plt.show()
    plt.close()
    
    return {
        'stage': stage_name,
        'responder_mean': responder.mean(),
        'non_responder_mean': non_responder.mean(),
        'p_value': pval,
        'effect_size': eff_size,
        'n_responder': len(responder),
        'n_non_responder': len(non_responder)
    }

# 3. 循环执行每个分期的分析
results = []
for stage in valid_stages:
    # 创建分期的数据子集
    stage_mask = adata.obs['gender'] == stage
    sub_adata = adata[stage_mask].copy()
    
    # 执行分析
    result = perform_analysis(sub_adata, stage)
    if result:
        results.append(result)

# 4. 汇总结果并保存
result_df = pd.DataFrame(results)
print("\n统计分析汇总:")
print(result_df.to_string(index=False))  # 优化表格显示

# 保存详细结果
result_df.to_csv(
    "Stratified_Analysis_Results.csv", 
    index=False,          # 不保存行索引
    float_format='%.4f'   # 保留4位小数
)
